# Histology QA Dashboard (Phase 4)

Interactive reconciliation of database counts vs manual reference.

1. Manual-vs-DB comparison table
2. PTC variant treemap
3. Surgery type breakdown
4. Parathyroid adenoma analysis
5. One-click full QA

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DB_PATH = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data/thyroid_master.duckdb')
con = duckdb.connect(str(DB_PATH), read_only=True)
print('Connected:', DB_PATH)

## 1) Manual vs DB Comparison Table

In [ ]:
manual = {
    'PTC (all)': 3000, 'FTC': 500, 'MTC': 155, 'ATC': 15,
    'Follicular adenoma': 925, 'Hurthle adenoma': 266,
    'MNG': 6000, 'Graves': 589, 'Hashimoto': 2168,
    'Parathyroid mentioned': 3332, 'Parathyroid adenoma': 94,
    'TGDC': 219, 'Hyalinizing trabecular': 9
}

db_queries = {
    'PTC (all)': "SELECT COUNT(DISTINCT research_id) FROM tumor_pathology WHERE histology_1_type = 'PTC'",
    'FTC': "SELECT COUNT(DISTINCT research_id) FROM tumor_pathology WHERE histology_1_type = 'FTC'",
    'MTC': "SELECT COUNT(DISTINCT research_id) FROM tumor_pathology WHERE histology_1_type = 'MTC'",
    'ATC': "SELECT COUNT(DISTINCT research_id) FROM tumor_pathology WHERE histology_1_type = 'ATC'",
    'Follicular adenoma': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_follicular_adenoma",
    'Hurthle adenoma': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_hurthle_adenoma",
    'MNG': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_mng",
    'Graves': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_graves",
    'Hashimoto': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_hashimoto",
    'Parathyroid mentioned': "SELECT COUNT(DISTINCT research_id) FROM parathyroid",
    'Parathyroid adenoma': "SELECT COUNT(DISTINCT research_id) FROM parathyroid WHERE is_parathyroid_adenoma",
    'TGDC': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_tgdc",
    'Hyalinizing trabecular': "SELECT COUNT(DISTINCT research_id) FROM benign_pathology WHERE is_hyalinizing_trabecular",
}

rows = []
for cat, m in manual.items():
    d = con.execute(db_queries[cat]).fetchone()[0]
    diff = d - m
    pct = abs(diff) / m * 100 if m else 0
    status = 'match' if pct <= 5 else ('close' if pct <= 15 else 'investigate')
    rows.append({'Category': cat, 'Manual': m, 'DB': d, 'Diff': diff, '% Diff': round(pct, 1), 'Status': status})

qa = pd.DataFrame(rows)
qa.style.apply(
    lambda r: [
        'background-color: #d4edda' if r['Status'] == 'match' else
        ('background-color: #fff3cd' if r['Status'] == 'close' else 'background-color: #f8d7da')
    ] * len(r),
    axis=1
)

## 2) PTC Variant Treemap

In [ ]:
ptc_var = con.execute('''
SELECT COALESCE(variant_standardized, 'Classic / NOS') AS variant,
       COUNT(DISTINCT research_id) AS n
FROM tumor_pathology
WHERE histology_1_type = 'PTC'
GROUP BY variant
ORDER BY n DESC
''').fetchdf()

fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('Set2', len(ptc_var))
ax.barh(ptc_var['variant'], ptc_var['n'], color=colors)
ax.invert_yaxis()
for i, (v, n) in enumerate(zip(ptc_var['variant'], ptc_var['n'])):
    ax.text(n + 10, i, str(n), va='center', fontsize=10)
ax.set_xlabel('Number of Patients')
ax.set_title('PTC Variant Distribution (n=3,278)')
plt.tight_layout()
plt.show()

## 3) Surgery Type Breakdown (Malignant vs Benign)

In [ ]:
surg_mal = con.execute('''
SELECT surgery_type_normalized AS surg_type, COUNT(*) AS n
FROM tumor_pathology GROUP BY 1 ORDER BY n DESC
''').fetchdf()
surg_mal['group'] = 'Malignant'

surg_ben = con.execute('''
SELECT surgery_type_normalized AS surg_type, COUNT(*) AS n
FROM benign_pathology GROUP BY 1 ORDER BY n DESC
''').fetchdf()
surg_ben['group'] = 'Benign'

surg = pd.concat([surg_mal, surg_ben], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, g in zip(axes, ['Malignant', 'Benign']):
    sub = surg[surg['group'] == g].sort_values('n', ascending=True)
    ax.barh(sub['surg_type'], sub['n'], color=sns.color_palette('muted'))
    ax.set_title(f'{g} Surgery Types')
    ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 4) Parathyroid Analysis

In [ ]:
para = con.execute('''
SELECT
    COALESCE(parathyroid_abnormality, 'none mentioned') AS abnormality,
    removal_intent,
    COUNT(DISTINCT research_id) AS n
FROM parathyroid
GROUP BY 1, 2
ORDER BY n DESC
''').fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie: abnormality type
abnorm = para.groupby('abnormality')['n'].sum().sort_values(ascending=False)
axes[0].pie(abnorm.values, labels=abnorm.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Parathyroid Abnormality Distribution')

# Bar: adenoma by intent
adenoma_intent = para[para['abnormality'] == 'adenoma'].sort_values('n', ascending=True)
axes[1].barh(adenoma_intent['removal_intent'], adenoma_intent['n'], color=['#2ecc71','#f39c12','#e74c3c'])
axes[1].set_title('Parathyroid Adenoma by Removal Intent')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"Total parathyroid adenoma (is_parathyroid_adenoma = True): "
      f"{con.execute('SELECT SUM(CASE WHEN is_parathyroid_adenoma THEN 1 ELSE 0 END) FROM parathyroid').fetchone()[0]}")

## 5) One-Click Full QA

In [ ]:
import subprocess, sys

def run_full_qa():
    root = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data')
    venv_python = root / '.venv' / 'bin' / 'python'
    scripts = [
        root / 'scripts' / '01_ingest_all_files.py',
        root / 'scripts' / '02_build_duckdb_full.py',
        root / 'scripts' / '03_research_views.py',
        root / 'scripts' / '04_publication_exports.py',
        root / 'scripts' / '05_histology_qa.py',
    ]
    for s in scripts:
        print(f'\n--- Running {s.name} ---')
        result = subprocess.run([str(venv_python), str(s)], capture_output=True, text=True)
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.returncode != 0:
            print('STDERR:', result.stderr[-500:])
            break
    print('\nFull QA pipeline complete.')

run_full_qa()